# IDD Temporal Val → video clips

**Vision-Based Lane Inference for Indian Roads · Phase 4**

One job: turn IDD Temporal Val (9.6 GB of frames) into a few hundred MB of MP4
clips on your Drive. Nothing is written to your laptop.

**Why this dataset.** IDD Temporal gives the 30 frames surrounding each
segmentation frame. Two parts of the pipeline are implemented but *unmeasured*,
because IDD-Lite holds independent frames rather than sequences:

* the Kalman smoother over lane geometry, and
* empirical lane inference — the idea that on an unmarked road the lanes are
  wherever traffic actually drives — which so far is validated only against
  synthetic trajectories.

Real sequences turn both from asserted into measured.

---

### You do exactly two things

**1.** `Runtime → Change runtime type → T4 GPU`

**2.** In section 2, paste one "Copy as cURL" string. Then `Runtime → Run all`.

Every cell below checks its own assumptions and says precisely what is wrong if
something fails, so there should be no second attempt. If a cell does fail, its
message names the fix.

## 1. Runtime, disk, Drive

In [ ]:
import shutil, subprocess, pathlib, sys

print(subprocess.run(['nvidia-smi','--query-gpu=name,memory.total','--format=csv,noheader'],
                     capture_output=True, text=True).stdout.strip() or 'no GPU (fine for this notebook)')

free = shutil.disk_usage('/content').free / 1e9
print('Colab local disk free : %.0f GB  (need ~25)' % free)
if free < 22:
    raise RuntimeError('Not enough local disk. Runtime > Disconnect and delete '
                       'runtime, then reconnect with a GPU runtime.')

from google.colab import drive
drive.mount('/content/drive')
DEST = pathlib.Path('/content/drive/MyDrive/ODP'); DEST.mkdir(parents=True, exist_ok=True)
dfree = shutil.disk_usage('/content/drive/MyDrive').free / 1e9
print('Drive free            : %.1f GB  (need ~1)' % dfree)
if dfree < 1.5:
    raise RuntimeError('Under 1.5 GB free on Drive. Clear some space first.')
print('\nready')

## 2. Download

The IDD token URL is **not** self-authenticating — fetched without your session
it returns the login page as HTML. Rather than have you dig out cookie values,
paste the request exactly as your browser makes it and this cell replays it.

### Get the cURL string (about 20 seconds)

1. Logged in, on https://idd.insaan.iiit.ac.in/dataset/download/
2. `Cmd+Option+I` → **Network** tab → leave it open
3. Click the **IDD Temporal Val** download link, then cancel the browser download
4. A request to `/dataset/download/d02bed47.../` appears in the Network list.
   Right-click it → **Copy** → **Copy as cURL**
5. Paste it between the triple quotes below

The cell parses out the URL, cookies and headers, replays them with curl, and
**pre-flights 512 bytes first** — so a wrong paste or an expired token fails in
about a second, not after a wasted transfer.

If the token has expired (they last 24 h) just regenerate it on that page and
re-copy.

In [ ]:
# Paste "Copy as cURL" between the triple quotes. That is the only input.
CURL = r"""

"""

# Fallback, if you would rather not use cURL: paste just the sessionid cookie
# value from DevTools > Application > Storage > Cookies.
SESSIONID = ''
FALLBACK_URL = 'https://idd.insaan.iiit.ac.in/dataset/download/d02bed47-80bf-4ab9-accb-adc4b4fe7b9c/'

EXPECT_GB = 9.6

import pathlib, re, shlex, subprocess

def parse_curl(text):
    """Pull the URL, cookie string and headers out of a DevTools cURL command."""
    text = text.strip()
    if not text:
        return None, None, []
    # DevTools wraps long commands with backslash-newline continuations.
    flat = re.sub(r'\\\s*\n\s*', ' ', text)
    try:
        tokens = shlex.split(flat)
    except ValueError as e:
        raise RuntimeError('Could not parse that cURL (%s). Paste it again, '
                           'unmodified, between the triple quotes.' % e)
    if not tokens or 'curl' not in tokens[0]:
        raise RuntimeError('That does not look like a cURL command - it should '
                           'start with "curl".')

    url, cookie, headers = None, None, []
    i = 1
    takes_value = {'-H','--header','-b','--cookie','-A','--user-agent','-e','--referer',
                   '-X','--request','-d','--data','--data-raw','--data-binary',
                   '-u','--user','--proxy','-x'}
    while i < len(tokens):
        t = tokens[i]
        if t in ('-H','--header'):
            headers.append(tokens[i+1]); i += 2
        elif t in ('-b','--cookie'):
            cookie = tokens[i+1]; i += 2
        elif t in ('-A','--user-agent'):
            headers.append('User-Agent: ' + tokens[i+1]); i += 2
        elif t in ('-e','--referer'):
            headers.append('Referer: ' + tokens[i+1]); i += 2
        elif t in takes_value:
            i += 2
        elif t.startswith('http'):
            url = t; i += 1
        else:
            i += 1
    # Some DevTools variants put cookies in a -H 'Cookie: ...' header instead.
    if cookie is None:
        for h in headers:
            if h.lower().startswith('cookie:'):
                cookie = h.split(':', 1)[1].strip()
    return url, cookie, [h for h in headers if not h.lower().startswith('cookie:')]

url, cookie, headers = parse_curl(CURL)

if url is None:
    if not SESSIONID.strip():
        raise RuntimeError(
            'No input. Paste a "Copy as cURL" string into CURL, or put your '
            'sessionid cookie value into SESSIONID. See the instructions above.')
    v = SESSIONID.strip().strip('"').strip("'")
    cookie = v if '=' in v else 'sessionid=' + v
    url = FALLBACK_URL
    headers = ['User-Agent: Mozilla/5.0',
               'Referer: https://idd.insaan.iiit.ac.in/dataset/download/']
    print('using the sessionid fallback')
else:
    print('parsed cURL OK')

if '/dataset/download/' not in url:
    raise RuntimeError('The URL parsed out is:\n  %s\nThat is not a dataset '
                       'download link. In the Network tab, copy the request to '
                       '/dataset/download/..., not a page load.' % url)
if not cookie:
    raise RuntimeError('No cookie found in that request. Make sure you are '
                       'logged in and that you copied the download request.')

print('url    :', url)
print('cookie : %d chars, keys %s'
      % (len(cookie), [c.split('=')[0].strip() for c in cookie.split(';')][:6]))
print('headers: %d' % len(headers))

BASE = ['curl', '-L', '-b', cookie]
for h in headers:
    BASE += ['-H', h]

RAW = pathlib.Path('/content/idd_raw'); RAW.mkdir(parents=True, exist_ok=True)
blob = RAW / 'temporal_val.bin'

# --- pre-flight -----------------------------------------------------------
# Piped through `head -c 512` deliberately: if the server ignores the Range
# request it would otherwise stream all 9.6 GB into memory. head closes the
# pipe, curl takes SIGPIPE and stops. Verified against the live IDD server --
# a real .tar.gz reads as gzip and an unauthenticated response reads as HTML,
# both inside one second.
import shlex as _shlex
_cmd = ' '.join(_shlex.quote(a) for a in BASE + ['-s', '--max-time', '45', '-r', '0-511', url])
probe = subprocess.run(_cmd + ' 2>/dev/null | head -c 512',
                       shell=True, capture_output=True).stdout
ARCHIVE = (probe[:2] == b'PK') or (probe[:2] == b'\x1f\x8b') or (probe[257:262] == b'ustar')
if not ARCHIVE:
    snippet = probe[:200].decode('utf-8', 'replace')
    raise RuntimeError(
        'The server did not return an archive, so the request is not authenticated '
        'or the token has expired.\n\nIt sent:\n  ' + snippet +
        '\n\nFix: confirm you are still logged in, regenerate the token at '
        'https://idd.insaan.iiit.ac.in/dataset/download/ , and re-copy the '
        'download request as cURL.')
print('\npre-flight OK - the server is returning an archive')

# --- download -------------------------------------------------------------
if blob.exists() and blob.stat().st_size > 0.9 * EXPECT_GB * 1e9:
    print('already have %.2f GB, skipping' % (blob.stat().st_size / 1e9))
else:
    print('fetching ~%s GB. Expect 20-60 min; progress prints below.' % EXPECT_GB, flush=True)
    r = subprocess.run(BASE + ['-C', '-', '--retry', '8', '--retry-delay', '15',
                               '--retry-all-errors', '-o', str(blob), url])
    if r.returncode != 0:
        raise RuntimeError('curl exited %d. Re-run this cell - it resumes from '
                           'where it stopped.' % r.returncode)

size = blob.stat().st_size / 1e9
print('\nhave %.2f GB' % size)
if size < 0.5 * EXPECT_GB:
    raise RuntimeError('Truncated at %.2f GB. Re-run this cell to resume.' % size)

## 3. Extract and find the sequences

IDD Temporal's internal layout is not documented, so this discovers it rather
than assuming: any directory holding several sortable image files is treated as
one sequence. If nothing is found, the cell prints the directory tree so the
layout can be seen at a glance.

In [ ]:
import pathlib, zipfile, tarfile, collections, cv2, shutil

TMP = pathlib.Path('/content/idd_temporal'); TMP.mkdir(parents=True, exist_ok=True)
blob = pathlib.Path('/content/idd_raw/temporal_val.bin')

def unpack(path, dest):
    # curl -o keeps no extension, so sniff the real container.
    if zipfile.is_zipfile(path):
        print('zip, extracting ...', flush=True)
        with zipfile.ZipFile(path) as z:
            z.extractall(dest)
    elif tarfile.is_tarfile(path):
        print('tar, extracting ...', flush=True)
        with tarfile.open(path) as t:
            t.extractall(dest)
    else:
        raise RuntimeError('Not a zip or tar. First bytes: '
                           + repr(path.read_bytes()[:120]))

if not any(TMP.iterdir()):
    unpack(blob, TMP)

# Nested archives happen; unpack one level of them too.
for inner in list(TMP.rglob('*')):
    if inner.is_file() and inner.suffix.lower() in ('.zip', '.tar', '.gz', '.tgz'):
        try:
            unpack(inner, inner.parent); inner.unlink()
        except Exception:
            pass

exts = ('.jpg', '.jpeg', '.png')
seqs = collections.defaultdict(list)
for f in TMP.rglob('*'):
    if f.suffix.lower() in exts:
        seqs[f.parent].append(f)
seqs = {d: sorted(v) for d, v in seqs.items() if len(v) >= 8}

if not seqs:
    print('No image sequences found. Directory tree, first 60 entries:\n')
    for n, q in enumerate(sorted(TMP.rglob('*'))):
        if n >= 60: break
        print('  ', q.relative_to(TMP), '(dir)' if q.is_dir() else q.stat().st_size)
    raise RuntimeError('Could not find sequences - the tree above shows the layout.')

lens = sorted(len(v) for v in seqs.values())
d0, v0 = sorted(seqs.items())[0]
im0 = cv2.imread(str(v0[0]))
print('sequences : %d' % len(seqs))
print('frames    : %d' % sum(lens))
print('per seq   : min %d, median %d, max %d' % (lens[0], lens[len(lens)//2], lens[-1]))
print('example   : %s' % d0.relative_to(TMP))
print('            %d frames, first four %s' % (len(v0), [p.name for p in v0[:4]]))
print('            frame size %dx%d' % (im0.shape[1], im0.shape[0]) if im0 is not None
      else '            FIRST FRAME UNREADABLE')

## 4. Encode clips

Written at 960 px wide, which is ample for the pipeline and keeps the archive
small. Clip names retain the drive and sequence identifiers so each clip can be
matched back to its IDD segmentation frame, and therefore to that frame's ground
truth.

In [ ]:
import cv2, pathlib, json, numpy as np

CLIPS = pathlib.Path('/content/idd_clips'); CLIPS.mkdir(parents=True, exist_ok=True)
MAX_CLIPS, FPS, WIDTH = 150, 10, 960

index, made, skipped = [], 0, 0
for d, frames in sorted(seqs.items()):
    if made >= MAX_CLIPS:
        break
    first = cv2.imread(str(frames[0]))
    if first is None:
        skipped += 1; continue
    h, w = first.shape[:2]
    scale = min(1.0, WIDTH / w)
    size = (int(w * scale), int(h * scale))
    name = '_'.join(d.relative_to(TMP).parts[-2:]).replace('/', '_') or ('seq%03d' % made)
    out = CLIPS / (name + '.mp4')

    vw = cv2.VideoWriter(str(out), cv2.VideoWriter_fourcc(*'mp4v'), FPS, size)
    if not vw.isOpened():
        raise RuntimeError('VideoWriter would not open - mp4v codec unavailable.')
    n = 0
    for f in frames:
        im = cv2.imread(str(f))
        if im is None:
            continue
        vw.write(cv2.resize(im, size, interpolation=cv2.INTER_AREA) if scale < 1.0 else im)
        n += 1
    vw.release()

    if n < 8 or not out.exists() or out.stat().st_size < 5000:
        out.unlink(missing_ok=True); skipped += 1; continue
    index.append({'clip': out.name, 'frames': n,
                  'width': size[0], 'height': size[1],
                  'source': str(d.relative_to(TMP))})
    made += 1
    if made % 25 == 0:
        print('  %d clips ...' % made, flush=True)

(CLIPS / 'index.json').write_text(json.dumps(index, indent=2))
print('\n%d clips written, %d sequences skipped' % (made, skipped))
if made == 0:
    raise RuntimeError('No clips produced.')
!du -sh /content/idd_clips

In [ ]:
# Confirm a clip is real motion, not the same frame repeated.
import cv2, numpy as np, matplotlib.pyplot as plt, pathlib
clip = sorted(pathlib.Path('/content/idd_clips').glob('*.mp4'))[0]
cap = cv2.VideoCapture(str(clip)); frames = []
while True:
    ok, f = cap.read()
    if not ok: break
    frames.append(f)
cap.release()
print(clip.name, '->', len(frames), 'frames read back')

fig, ax = plt.subplots(1, 4, figsize=(18, 3.4))
for i, k in enumerate(np.linspace(0, len(frames)-1, 4).astype(int)):
    ax[i].imshow(frames[k][:, :, ::-1]); ax[i].axis('off'); ax[i].set_title('frame %d' % k)
plt.tight_layout(); plt.show()

d = float(np.mean([np.abs(frames[i].astype(int) - frames[i+1].astype(int)).mean()
                   for i in range(min(10, len(frames)-1))]))
print('mean inter-frame difference: %.2f' % d)
print('near 0 would mean duplicated frames; a few units means real motion.')

## 5. Save to Drive and free the space

In [ ]:
import subprocess, shutil, pathlib
dest = pathlib.Path('/content/drive/MyDrive/ODP')
subprocess.run(['tar','-czf','/content/idd_clips.tar.gz','-C','/content','idd_clips'], check=True)
pack = pathlib.Path('/content/idd_clips.tar.gz')
shutil.copy(pack, dest / 'idd_clips.tar.gz')
mb = pack.stat().st_size / 1e6
print('saved %s  (%.0f MB)' % (dest / 'idd_clips.tar.gz', mb))
assert (dest / 'idd_clips.tar.gz').stat().st_size == pack.stat().st_size, 'copy to Drive incomplete'

shutil.rmtree('/content/idd_temporal', ignore_errors=True)
shutil.rmtree('/content/idd_raw', ignore_errors=True)
!df -h /content | tail -1
print('\nDone. idd_clips.tar.gz is on Drive; the 9.6 GB has been freed.')